Connected to .venv (Python 3.10.19)

In [ ]:
import keras
import tensorflow as tf
from preprocessing import train_ds, valid_ds, TRAIN_SIZE, VALID_SIZE, dice_metric, combined_loss, dice_ET, dice_TC, dice_WT

tf.random.set_seed(0)

he_init=keras.initializers.HeNormal()

elu_act=keras.activations.elu

l2_reg=keras.regularizers.l2(1e-4)

inputs=keras.layers.Input(shape=(128,128,4))

# Encoder which downscales the image
x=keras.layers.Conv2D(filters=32, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(inputs)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=32, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
block_1_output=x
x=keras.layers.MaxPool2D((2,2))(x)

skip_layer_1=keras.layers.Conv2D(filters=64, kernel_size=(1,1), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.Conv2D(filters=64, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=64, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.add([x,skip_layer_1])
block_2_output=x
x=keras.layers.MaxPool2D((2,2))(x)

skip_layer_2=keras.layers.Conv2D(filters=128, kernel_size=(1,1), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.Conv2D(filters=128, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=128, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.add([x,skip_layer_2])
block_3_output=x
x=keras.layers.MaxPool2D((2,2))(x)

skip_layer_3=keras.layers.Conv2D(filters=256, kernel_size=(1,1), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.Conv2D(filters=256, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=256, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.add([x,skip_layer_3])
block_4_output=x
x=keras.layers.MaxPool2D((2,2))(x)

x=keras.layers.Conv2D(512, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Dropout(0.1)(x)

x=keras.layers.Conv2D(512, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Dropout(0.1)(x)

bottleneck=x

# Decoder which upscales the images

x=keras.layers.Conv2DTranspose(filters=256,kernel_size=(2,2),strides=2, padding="same", kernel_initializer=he_init, kernel_regularizer=l2_reg)(bottleneck)
x=keras.layers.Concatenate()([x,block_4_output])
x=keras.layers.Dropout(0.2)(x)

block_1_decoder_output=keras.layers.Conv2D(256, kernel_size=(1,1), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)

x=keras.layers.Conv2D(filters=256, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=256, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.add([x, block_1_decoder_output])

x=keras.layers.Conv2DTranspose(filters=128,kernel_size=(2,2),strides=2, padding="same", kernel_initializer=he_init, kernel_regularizer=l2_reg)(x)
x=keras.layers.Concatenate()([x,block_3_output])
x=keras.layers.Dropout(0.2)(x)

block_2_decoder_output=keras.layers.Conv2D(128, kernel_size=(1,1), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)

x=keras.layers.Conv2D(filters=128, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=128, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.add([x, block_2_decoder_output])

x=keras.layers.Conv2DTranspose(filters=64,kernel_size=(2,2),strides=2, padding="same", kernel_initializer=he_init, kernel_regularizer=l2_reg)(x)
x=keras.layers.Concatenate()([x,block_2_output])
x=keras.layers.Dropout(0.2)(x)

block_3_decoder_output=keras.layers.Conv2D(64, kernel_size=(1,1), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)

x=keras.layers.Conv2D(filters=64, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=64, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.add([x, block_3_decoder_output])

x=keras.layers.Conv2DTranspose(filters=32,kernel_size=(2,2),strides=2, padding="same", kernel_initializer=he_init, kernel_regularizer=l2_reg)(x)
x=keras.layers.Concatenate()([x,block_1_output])
x=keras.layers.Dropout(0.2)(x)

block_4_decoder_output=keras.layers.Conv2D(32, kernel_size=(1,1), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)

x=keras.layers.Conv2D(filters=32, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=32, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.add([x, block_4_decoder_output])

outputs=keras.layers.Conv2D(4, kernel_size=(1,1), activation='softmax')(x)

model=keras.Model(inputs, outputs)

adam=keras.optimizers.Adam(1e-4, clipnorm=1.0)

sgd=keras.optimizers.SGD(1e-3,momentum=0.95, nesterov=True)

model.compile(
    optimizer=adam,
    loss=combined_loss,
    metrics=[dice_metric, dice_ET, dice_TC, dice_WT]
)

earlyStop_cb=keras.callbacks.EarlyStopping(
    monitor='val_dice_metric',
    patience=10,
    verbose=1,
    restore_best_weights=True,
    mode='max'
)

lrPlateau_cb=keras.callbacks.ReduceLROnPlateau(
    monitor='val_dice_metric',
    patience=3,
    factor=0.5,
    verbose=1,
    min_lr=5e-7,
    mode='max'
)

history=model.fit(
    train_ds,
    batch_size=32,
    epochs=13,
    callbacks=[earlyStop_cb, lrPlateau_cb],
    validation_data=valid_ds,
    steps_per_epoch = 1500,
    validation_steps = 100
)

model.save('Saved Models/Seg_Adam_lr1e-4_LrPlateau_CombinedLoss_DiceMetric_DicePerClass.keras')

2026-05-01 21:22:36.647456: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M5 Pro
2026-05-01 21:22:36.647481: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 24.00 GB
2026-05-01 21:22:36.647484: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 8.88 GB
2026-05-01 21:22:36.647499: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-05-01 21:22:36.647507: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Epoch 1/13


2026-05-01 21:22:38.897359: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2026-05-01 21:22:50.727329: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:5: Filling up shuffle buffer (this may take a while): 240 of 256
2026-05-01 21:22:51.356514: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:480] Shuffle buffer filled.


1500/1500 ━━━━━━━━━━━━━━━━━━━━ 1065s 698ms/step - dice_et: 0.5205 - dice_metric: 0.4751 - dice_tc: 0.5219 - dice_wt: 0.6174 - loss: 1.0117 - val_dice_et: 0.6312 - val_dice_metric: 0.5671 - val_dice_tc: 0.6632 - val_dice_wt: 0.7390 - val_loss: 0.7722 - learning_rate: 1.0000e-04
Epoch 2/13
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 706s 470ms/step - dice_et: 0.6899 - dice_metric: 0.6050 - dice_tc: 0.7353 - dice_wt: 0.7569 - loss: 0.6753 - val_dice_et: 0.6881 - val_dice_metric: 0.5741 - val_dice_tc: 0.7005 - val_dice_wt: 0.7341 - val_loss: 0.6465 - learning_rate: 1.0000e-04
Epoch 3/13
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 705s 470ms/step - dice_et: 0.7240 - dice_metric: 0.6298 - dice_tc: 0.7729 - dice_wt: 0.7764 - loss: 0.5772 - val_dice_et: 0.7630 - val_dice_metric: 0.5822 - val_dice_tc: 0.7844 - val_dice_wt: 0.7602 - val_loss: 0.6017 - learning_rate: 1.0000e-04
Epoch 4/13
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 705s 470ms/step - dice_et: 0.7411 - dice_metric: 0.6432 - dice_tc: 0.7889 - dice_wt: 0.7838 - loss: 0.52